# SparkX-Loop v0.2 — L4 Inference, Telemetry & OpenCode Go Evaluation Suite

This notebook tests your mid-training or completed **60-Effective-Layer Looped Recursive + 3-Tier MoE-Engram** checkpoint on a **Google Colab L4 GPU (24 GB VRAM)** without disturbing your running G4 training session.

- **Default Target Checkpoint Repo**: [`TassP0lar/Spark-X2.6-4B-Loop-mHC-R24-step01850`](https://huggingface.co/TassP0lar/Spark-X2.6-4B-Loop-mHC-R24-step01850)
- **Base Architecture**: `XHToken/Spark-X2.6-4B` (`36 physical layers -> 60 virtual layers` via Sublayer Dual-mHC + 7 Backbone MoE-Engrams = 112 Memory Experts)
- **OpenCode Go Integration**: Uses your **OpenCode Go Coding Plan** (`https://opencode.ai/zen/go/v1`) with `OPENCODE_API_KEY` for automated blind LLM-as-a-Judge grading and side-by-side coding/reasoning verification.

### Quick Start on Colab L4
1. Select **Runtime > Change runtime type > L4 GPU**.
2. Ensure `HF_TOKEN` (or `HF_TOKEM`) and `OPENCODE_API_KEY` (or `OPENCODE_API`) are enabled in **Colab Secrets** (left sidebar key icon).
3. Click **Runtime > Run all**.

In [ ]:
# === Cell 1: Environment Bootstrap & GitHub Repo Sync (L4 GPU) ===
import os, sys, subprocess, shutil
from pathlib import Path

os.environ['HF_HUB_DISABLE_INTERACTIVE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PIP_NO_INPUT'] = '1'

REPO_URL = 'https://github.com/BaiTian6641/Spark-X2.6-Recursion.git'
REPO_DIR = Path('/content/Spark-X2.6-Recursion')

if Path.cwd().name == 'Spark-X2.6-Recursion' and (Path.cwd() / 'sparkx_loop_core.py').exists():
    REPO_DIR = Path.cwd()
elif Path('/storage/emulated/10/Documents/Spark_X2.6_Loop/sparkx_loop_core.py').exists():
    REPO_DIR = Path('/storage/emulated/10/Documents/Spark_X2.6_Loop')
else:
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', 'main'], check=False)
        subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', 'origin/main'], check=False)
    os.chdir(REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.1', 'accelerate', 'safetensors', 'huggingface_hub', 'openai', 'datasets'
], check=True)

def get_secret(*names, default=''):
    for name in names:
        val = os.environ.get(name, '').strip()
        if val:
            return val
    try:
        from google.colab import userdata
        for name in names:
            try:
                val = userdata.get(name)
                if val and str(val).strip():
                    os.environ[name] = str(val).strip()
                    return str(val).strip()
            except Exception:
                pass
    except Exception:
        pass
    return default

HF_TOKEN = get_secret('HF_TOKEN', 'HF_TOKEM')
OPENCODE_API_KEY = get_secret('OPENCODE_API_KEY', 'OPENCODE_API', 'OPENCODE_GO_API_KEY', 'OPENCODE_KEY')

import torch
assert torch.cuda.is_available(), 'CUDA GPU required! Select L4 GPU in Colab Runtime settings.'
gpu_name = torch.cuda.get_device_name(0)
vram_gib = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f'GPU Ready: {gpu_name} ({vram_gib:.1f} GiB VRAM)')
print(f'HF_TOKEN loaded: {bool(HF_TOKEN)} | OPENCODE_API_KEY loaded: {bool(OPENCODE_API_KEY)}')

In [ ]:
# === Cell 2: Download & Load TassP0lar/Spark-X2.6-4B-Loop-mHC-R24-step02100 ===
import json
import torch
from huggingface_hub import snapshot_download
from safetensors.torch import load_file
from transformers import AutoTokenizer, AutoModelForCausalLM
from sparkx_loop_core import (
    LoopSettings, SparkXLoop, attach_lora, patch_spark_attention_for_long_context
)

# Change this anytime to test newer checkpoints (e.g., step02100 or final R24)
CHECKPOINT_REPO = os.environ.get('CHECKPOINT_REPO', 'TassP0lar/Spark-X2.6-4B-Loop-mHC-R24')
STAGE_SUBFOLDER = os.environ.get('LOOP_STAGE', 'R24')

print(f'Downloading checkpoint from: {CHECKPOINT_REPO} ...')
ckpt_root = Path(snapshot_download(
    repo_id=CHECKPOINT_REPO,
    token=HF_TOKEN or None,
    allow_patterns=['loop_config.json', 'trainable.safetensors', f'stages/{STAGE_SUBFOLDER}/*', '*.json']
))
ckpt_dir = (ckpt_root / 'stages' / STAGE_SUBFOLDER) if (ckpt_root / 'stages' / STAGE_SUBFOLDER / 'loop_config.json').exists() else ckpt_root
meta = json.loads((ckpt_dir / 'loop_config.json').read_text())
print('Loaded loop_config.json:', json.dumps({
    k: meta.get(k) for k in [
        'stage', 'ExportTag', 'train_mode', 'loop_start', 'loop_end', 'loop_passes',
        'engram_buckets', 'engram_dim', 'engram_experts', 'engram_top_k', 'backbone_engram_stride'
    ] if k in meta
}, indent=2))

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
base_id = meta.get('base_id', 'XHToken/Spark-X2.6-4B')
base_rev = meta.get('base_revision', None)

if 'base' not in globals() or 'tok' not in globals():
    print(f'Loading base model & tokenizer ({base_id}) in {dtype} ...')
    tok = AutoTokenizer.from_pretrained(base_id, revision=base_rev, trust_remote_code=True, token=HF_TOKEN or None)
    base = AutoModelForCausalLM.from_pretrained(
        base_id, revision=base_rev, trust_remote_code=True,
        torch_dtype=dtype, device_map={'': 'cuda:0'}, token=HF_TOKEN or None, attn_implementation='eager'
    )
    base.config.use_cache = False
    if meta.get('lora_rank', 0) > 0:
        attach_lora(base, start=4, end=36, rank=meta['lora_rank'], alpha=meta['lora_alpha'])

settings = LoopSettings(
    start=4,
    end=28,
    passes=meta.get('loop_passes', 2),
    eng_buckets=meta.get('engram_buckets', 16384),
    eng_dim=meta.get('engram_dim', 64),
    eng_experts=meta.get('engram_experts', 16),
    eng_top_k=meta.get('engram_top_k', 2),
    eng_expert_rank=meta.get('engram_expert_rank', 16),
    eng_shared_rank=meta.get('engram_shared_rank', 16),
    backbone_engram_stride=meta.get('backbone_engram_stride', 4),
    checkpoint_layers=False,
    dynamic_mhc=meta.get('dynamic_mhc', True),
    sublayer_mhc=meta.get('sublayer_mhc', True),
    pass_adapter_rank=meta.get('pass_adapter_rank', 16),
)

net = SparkXLoop(base, settings, patch_sdpa=True).to('cuda:0')
print('Loading trained weights from trainable.safetensors ...')
state = load_file(str(ckpt_dir / 'trainable.safetensors'), device='cpu')
missing, unexpected = net.load_state_dict(state, strict=False)
del state
torch.cuda.empty_cache()

assert not unexpected, f'Unexpected keys in checkpoint: {unexpected[:10]}'
net.set_loop(
    meta.get('loop_start', 4),
    meta.get('loop_end', 28),
    meta.get('loop_passes', 2),
    use_mhc=meta.get('mhc_enabled', True),
    use_engram=meta.get('engram_enabled', True),
)
net.eval()

alloc_gib = torch.cuda.memory_allocated() / (1024 ** 3)
print(f'Successfully loaded {CHECKPOINT_REPO}!')
print(f'Effective Depth: {net.total_virtual_layers} virtual layers (Physical: {net.n_layers} layers)')
print(f'Frozen keys inherited from base (not in trainable.safetensors): {missing}')
print(f'L4 GPU VRAM Allocated: {alloc_gib:.2f} GiB')

In [ ]:
# === Cell 3: Inspect Learned Sublayer Dual-mHC Gates & 3-Tier MoE-Engram Telemetry ===
import torch

print('=== 1. Sublayer Dual-mHC Recurrent Gate Profile (Layers 4..27) ===')
print(f'{"Layer":>5} | {"Type":<18} | {"attn_gate (tanh)":>16} | {"ffn_gate (tanh)":>15} | {"Pass1 Attn Keep":>15} | {"Pass1 FFN Keep":>14}')
print('-' * 94)

attn_gates, ffn_gates = [], []
with torch.no_grad():
    for idx in range(net.loop_start, net.loop_end):
        ctrl = net.controllers[str(idx)]
        ltype = net.base.model.layers[idx].layer_type
        ag = float(torch.tanh(ctrl.extra_gain.float()).item())
        fg = float(torch.tanh(ctrl.ffn_extra_gain.float()).item())
        attn_gates.append(abs(ag))
        ffn_gates.append(abs(fg))
        # Static + Pass-1 routing bias diagonal keep probability
        m_attn = ctrl.mix_logits.float() + ctrl.pass_routing_bias[1, 0, 4:].reshape(2, 2).float()
        m_ffn = ctrl.ffn_mix_logits.float() + ctrl.pass_routing_bias[1, 1, 4:].reshape(2, 2).float()
        p_attn = float(ctrl.mixing_matrix(base_logits=m_attn)[0, 0].item())
        p_ffn = float(ctrl.mixing_matrix(base_logits=m_ffn)[0, 0].item())
        print(f'{idx:5d} | {ltype:<18} | {ag:+16.4f} | {fg:+15.4f} | {p_attn:15.4f} | {p_ffn:14.4f}')

print('-' * 94)
print(f'Mean |attn_gate| = {sum(attn_gates)/len(attn_gates):.4f} | Mean |ffn_gate| = {sum(ffn_gates)/len(ffn_gates):.4f}\n')

print('=== 2. 3-Tier Memory-Tiered MoE-Engram Modules (Prelude + 6 Macro-Block Anchors) ===')
print(f'{"Location":<18} | {"sig(gain)":>10} | {"||project||_F":>13} | {"||shared_up||_F":>15} | {"Expert Load Min/Max (16 exp)":>28}')
print('-' * 94)
with torch.no_grad():
    all_engrams = [('Prelude (Layer 3)', net.engram)] + [
        (f'Backbone (Layer {k})', mod) for k, mod in net.backbone_engrams.items()
    ]
    for label, mod in all_engrams:
        g = float(torch.sigmoid(mod.gain.float()).item())
        p_norm = float(mod.project.weight.float().norm().item())
        s_norm = float(mod.shared_up.weight.float().norm().item())
        ema = mod.expert_load_ema.float()
        ema_min, ema_max = float(ema.min().item()), float(ema.max().item())
        print(f'{label:<18} | {g:10.4f} | {p_norm:13.4f} | {s_norm:15.4f} | {ema_min:12.4f} .. {ema_max:.4f}')

In [ ]:
# === Cell 4: Live-Streaming Per-Visit KV-Cached 60-Layer Generation Engine (4096+ Token Cap & <think> Reasoning) ===
import sys
import time
import torch
import torch.nn.functional as F
from transformers import TextStreamer
from transformers.cache_utils import DynamicCache

DEFAULT_MAX_NEW_TOKENS = 4096

@torch.no_grad()
def generate_sparkx_loop(
    messages,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    enable_thinking: bool = True,
    reasoning_effort: str = 'high',
    temperature: float = 0.0,
    top_p: float = 0.9,
    repetition_penalty: float = 1.05,
    stream: bool = True,
):
    """Generate from SparkXLoop using per-visit KV cache (_VisitCache) across 60 virtual layers
    with real-time token streaming and guaranteed <think>...</think> reasoning activation.
    """
    net.eval()
    try:
        prompt_ids = tok.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            enable_thinking=enable_thinking,
            reasoning_effort=reasoning_effort,
            return_tensors='pt',
        )
    except TypeError:
        prompt_ids = tok.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
        )
    if hasattr(prompt_ids, 'input_ids'):
        prompt_ids = prompt_ids.input_ids
    elif isinstance(prompt_ids, dict):
        prompt_ids = prompt_ids['input_ids']
    input_ids = prompt_ids.to('cuda:0')

    # Ensure <think> prefix is active whenever enable_thinking=True
    prompt_decoded = tok.decode(input_ids[0], skip_special_tokens=False)
    if enable_thinking and '<think>' not in prompt_decoded.split('<|im_start|>assistant')[-1]:
        think_prefix = tok('<think>\n', add_special_tokens=False, return_tensors='pt')['input_ids'].to('cuda:0')
        input_ids = torch.cat((input_ids, think_prefix), dim=-1)

    try:
        cache = DynamicCache(config=net._make_loop_cache_config())
    except TypeError:
        cache = DynamicCache()

    streamer = TextStreamer(tok, skip_prompt=False, skip_special_tokens=False) if stream else None
    if stream and enable_thinking:
        sys.stdout.write('<think>\n')
        sys.stdout.flush()

    def sample_next(logits_step, generated_seq):
        logits_step = logits_step[:, -1, :].float()
        if repetition_penalty != 1.0 and generated_seq.numel() > 0:
            unique_tokens = generated_seq[0].unique()
            score = logits_step[0, unique_tokens]
            score = torch.where(score < 0, score * repetition_penalty, score / repetition_penalty)
            logits_step[0, unique_tokens] = score
        if temperature <= 0.0:
            return logits_step.argmax(dim=-1, keepdim=True)
        probs = F.softmax(logits_step / temperature, dim=-1)
        if top_p < 1.0:
            sorted_probs, sorted_indices = torch.sort(probs, descending=True)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            mask = (cum_probs - sorted_probs) > top_p
            sorted_probs[mask] = 0.0
            sorted_probs /= sorted_probs.sum(dim=-1, keepdim=True)
            next_local = torch.multinomial(sorted_probs, num_samples=1)
            return sorted_indices.gather(-1, next_local)
        return torch.multinomial(probs, num_samples=1)

    t0 = time.perf_counter()
    out = input_ids
    logits = net.forward_logits(out, past_key_values=cache)
    next_id = sample_next(logits, out[:, input_ids.shape[-1]:])
    prev_tok = out[:, -2:]
    out = torch.cat((out, next_id), dim=-1)
    if streamer is not None:
        streamer.put(next_id.cpu())

    if tok.eos_token_id is None or not torch.all(next_id == tok.eos_token_id):
        for _ in range(max_new_tokens - 1):
            logits = net.forward_logits(next_id, past_key_values=cache, prev_ids=prev_tok)
            prev_tok = out[:, -2:]
            next_id = sample_next(logits, out[:, input_ids.shape[-1]:])
            out = torch.cat((out, next_id), dim=-1)
            if streamer is not None:
                streamer.put(next_id.cpu())
            if tok.eos_token_id is not None and torch.all(next_id == tok.eos_token_id):
                break

    if streamer is not None:
        streamer.end()

    elapsed = time.perf_counter() - t0
    gen_ids = out[0, input_ids.shape[-1]:]
    raw_text = tok.decode(gen_ids, skip_special_tokens=False)
    clean_text = tok.decode(gen_ids, skip_special_tokens=True)

    thinking, answer = '', clean_text
    if '</think>' in raw_text:
        parts = raw_text.split('</think>', 1)
        thinking = parts[0].replace('<think>', '').strip()
        answer = parts[1].replace('<|im_end|>', '').replace('<|endoftext|>', '').strip()
    elif enable_thinking:
        thinking = clean_text.replace('<think>', '').strip()
        answer = ''

    return {
        'raw_text': raw_text,
        'thinking': thinking,
        'answer': answer,
        'num_tokens': int(gen_ids.numel()),
        'tok_per_sec': float(gen_ids.numel() / max(elapsed, 1e-6)),
        'elapsed_sec': elapsed,
    }

print(f'Live-streaming 60-layer generation engine ready (default max_new_tokens={DEFAULT_MAX_NEW_TOKENS}, enable_thinking=True).')

In [ ]:
# === Cell 5: Multi-Domain Live-Streaming Benchmark (4096 Token Cap + <think> Reasoning Enabled) ===
TEST_SUITE = [
    {
        'name': '1. PhD / University Physics & Thermodynamics',
        'messages': [
            {'role': 'user', 'content': (
                'A monatomic ideal gas undergoes a reversible polytropic process PV^n = constant '
                'from state (P1, V1, T1) to (P2, V2, T2). Derive the molar heat capacity C_n in terms '
                'of the universal gas constant R and polytropic index n, and find the exact range of n '
                'for which the molar heat capacity C_n is negative.'
            )}
        ],
        'max_new_tokens': 4096,
    },
    {
        'name': '2. Mathematical Reasoning (GSM8K / Competition Math)',
        'messages': [
            {'role': 'user', 'content': (
                'A research lab has two servers, Alpha and Beta. Together they can process a genomic '
                'dataset in 12 hours. If Alpha works alone for 4 hours and then Beta works alone for '
                '9 hours, they finish 50% of the dataset. How many hours would it take Alpha alone '
                'to process the entire dataset? Show step-by-step reasoning.'
            )}
        ],
        'max_new_tokens': 4096,
    },
    {
        'name': '3. Algorithmic Coding (Python)',
        'messages': [
            {'role': 'user', 'content': (
                'Write an efficient Python function `longest_balanced_subarray(nums: list[int]) -> int` '
                'that returns the length of the longest contiguous subarray where the sum of elements '
                'is divisible by k=7, running in O(n) time and O(k) space. Include a brief complexity proof.'
            )}
        ],
        'max_new_tokens': 4096,
    },
    {
        'name': '4. Multi-Turn Conversational Coherence & Deep Deliberation',
        'messages': [
            {'role': 'user', 'content': 'We are designing a recursive transformer where layers [4..28) execute twice using doubly-stochastic 2x2 Sinkhorn gates.'},
            {'role': 'assistant', 'content': '<think>The user is describing a looped recursive transformer with 24 middle layers unrolled for 2 passes and 2-stream manifold-constrained hyper-connections (mHC).</think>That architecture gives you 60 effective layers from 36 physical layers while bounding residual stream norms via the Birkhoff polytope. What aspect are we optimizing next?'},
            {'role': 'user', 'content': 'Why did our mean per-layer |ffn_gate| settle around 0.102 when all 24 layers were looped, compared to 0.246 when only 12 layers were looped?'}
        ],
        'max_new_tokens': 4096,
    },
]

SUITE_RESULTS = []
for item in TEST_SUITE:
    print('=' * 96)
    print(f"TEST: {item['name']} (max_new_tokens={item['max_new_tokens']}, enable_thinking=True, stream=True)")
    print('-' * 96)
    res = generate_sparkx_loop(
        item['messages'],
        max_new_tokens=item['max_new_tokens'],
        enable_thinking=True,
        reasoning_effort='high',
        temperature=0.0,
        stream=True,
    )
    SUITE_RESULTS.append({**item, **res})
    print(f"\n[Completed: {res['num_tokens']} tokens in {res['elapsed_sec']:.2f}s ({res['tok_per_sec']:.1f} tok/s) | <think> chars: {len(res['thinking'])} | answer chars: {len(res['answer'])}]\n")

In [ ]:
# === Cell 6: Live Architecture Ablation (60-Layer Full vs Loop-Only vs 36-Layer Single-Pass) ===
import math
import torch

ablation_messages = [
    {'role': 'user', 'content': 'Prove that if p > 3 is a prime number, then p^2 - 1 is always divisible by 24.'},
    {'role': 'assistant', 'content': (
        '<think>Let p > 3 be prime. Since p is prime and greater than 3, p is odd and not divisible by 3. '
        'Notice p^2 - 1 = (p - 1)(p + 1). Since p is odd, both p - 1 and p + 1 are two consecutive even integers, '
        'so one is divisible by 2 and the other by 4, meaning their product is divisible by 8. '
        'Also, among three consecutive integers p - 1, p, p + 1, one is divisible by 3; since p is prime > 3, '
        'either p - 1 or p + 1 is divisible by 3. Since gcd(3, 8) = 1, (p - 1)(p + 1) is divisible by 24.</think>'
        'Since $p > 3$ is prime, $p^2 - 1 = (p - 1)(p + 1)$ is the product of two consecutive even integers '
        '(giving a factor of $2 \\times 4 = 8$) and one of $p - 1, p + 1$ must be a multiple of $3$ '
        '(since $p$ is not divisible by $3$). Because $\\gcd(8, 3) = 1$, $24 \\mid (p^2 - 1)$.'
    )}
]

full_ids = tok.apply_chat_template(ablation_messages, tokenize=True, return_tensors='pt')
if hasattr(full_ids, 'input_ids'):
    full_ids = full_ids.input_ids
elif isinstance(full_ids, dict):
    full_ids = full_ids['input_ids']
full_ids = full_ids.to('cuda:0')

modes = [
    ('60-Layer Full (R24 Loop + Sublayer Dual-mHC + 7 MoE-Engrams)', 4, 28, 2, True, True),
    ('60-Layer Loop Only (R24 Loop + Dual-mHC, MoE-Engrams Disabled)', 4, 28, 2, True, False),
    ('36-Layer Single-Pass + MoE-Engrams (Recurrent Loop Disabled)', 4, 28, 1, False, True),
    ('36-Layer Pure Backbone (Both Loop & MoE-Engrams Disabled)', 4, 28, 1, False, False),
]

print(f'{"Architecture Mode":<62} | {"Eff. Layers":>11} | {"Loss (NLL)":>10} | {"Perplexity":>10}')
print('-' * 102)
with torch.no_grad():
    for title, s, e, p, use_mhc, use_eng in modes:
        net.set_loop(s, e, p, use_mhc=use_mhc, use_engram=use_eng)
        loss = float(net(full_ids, labels=full_ids)['loss'].item())
        ppl = math.exp(min(loss, 20.0))
        print(f'{title:<62} | {net.total_virtual_layers:11d} | {loss:10.4f} | {ppl:10.4f}')

# Restore full 60-layer R24 + MoE-Engram configuration
net.set_loop(4, 28, 2, use_mhc=True, use_engram=True)

In [ ]:
# === Cell 7: OpenCode Go Coding Plan Judge (https://opencode.ai/zen/go/v1) ===
# Queries /models on your OpenCode Go subscription, handles reasoning_content + content, and grades outputs.
from openai import OpenAI

OPENCODE_API_KEY = get_secret('OPENCODE_API_KEY', 'OPENCODE_API', 'OPENCODE_GO_API_KEY', 'OPENCODE_KEY')
OPENCODE_GO_URLS = [
    'https://opencode.ai/zen/go/v1',
    'https://opencode.ai/zen/v1',
    'https://opencode.ai/inference/openai/v1',
]
PREFERRED_MODELS = [
    'deepseek-v4.1-flash',
    'deepseek-v4-pro',
    'mimo-v2.6-pro',
    'mimo-v2.6-flash',
    'claude-haiku-5.5',
    'qwen3.8-max',
    'qwen3.8-flash',
    'glm-5.3',
    'glm-5.3-flash',
    'kimi-k3.1-pro',
    'kimi-k3',
    'minimax-m3.1-flash',
    'gpt-6-luna',
]

def discover_opencode_go():
    if not OPENCODE_API_KEY:
        print('WARNING: OPENCODE_API_KEY is empty. Check Colab Secrets (key icon on left sidebar).')
        return []
    endpoints = []
    for base_url in OPENCODE_GO_URLS:
        client = OpenAI(api_key=OPENCODE_API_KEY, base_url=base_url, timeout=60.0, max_retries=1)
        try:
            resp_models = client.models.list()
            live_ids = [m.id for m in getattr(resp_models, 'data', []) if getattr(m, 'id', None)]
            if live_ids:
                print(f'Discovered {len(live_ids)} live models at {base_url}: {live_ids[:12]}')
                # Prioritize preferred models that exist in live_ids, then remaining live_ids
                ordered = [m for m in PREFERRED_MODELS if m in live_ids] + [m for m in live_ids if m not in PREFERRED_MODELS]
                endpoints.append((base_url, client, ordered))
                continue
        except Exception as e:
            print(f'Note: /models probe on {base_url}: {type(e).__name__}: {str(e)[:120]}')
        endpoints.append((base_url, client, list(PREFERRED_MODELS)))
    return endpoints

ACTIVE_ENDPOINTS = discover_opencode_go()

def call_opencode_go(prompt: str, max_tokens: int = 2048):
    if not ACTIVE_ENDPOINTS:
        return None, 'OPENCODE_API_KEY secret not set in Colab Secrets.'
    last_err = None
    for base_url, client, candidate_models in ACTIVE_ENDPOINTS:
        for model_id in candidate_models:
            try:
                resp = client.chat.completions.create(
                    model=model_id,
                    messages=[
                        {'role': 'system', 'content': 'You are a concise PhD STEM and software engineering evaluator. Be direct.'},
                        {'role': 'user', 'content': prompt},
                    ],
                    max_tokens=max_tokens,
                    temperature=0.1,
                )
                msg = resp.choices[0].message
                text = (getattr(msg, 'content', None) or '').strip()
                if not text:
                    # Fallback if reasoning model placed output in reasoning_content / reasoning
                    text = (getattr(msg, 'reasoning_content', None) or getattr(msg, 'reasoning', None) or '').strip()
                if text:
                    return f'{model_id} @ {base_url}', text
            except Exception as e:
                last_err = f'{model_id} @ {base_url} -> {type(e).__name__}: {str(e)[:160]}'
                continue
    return None, f'All candidate models failed. Last error: {last_err}'

if OPENCODE_API_KEY and SUITE_RESULTS:
    print('\nRunning OpenCode Go Coding Plan LLM-as-a-Judge across test suite outputs...\n')
    for item in SUITE_RESULTS:
        user_q = item['messages'][-1]['content']
        judge_prompt = (
            f"Evaluate the following model response to the prompt below.\n\n"
            f"[PROMPT]:\n{user_q}\n\n"
            f"[MODEL INTERNAL <think>]:\n{item['thinking'][:4000]}\n\n"
            f"[MODEL FINAL ANSWER]:\n{item['answer'][:4000]}\n\n"
            f"Provide: (1) Score out of 10 for correctness, "
            f"(2) Score out of 10 for reasoning clarity, and (3) a brief 2-sentence critique."
        )
        used_endpoint, verdict = call_opencode_go(judge_prompt)
        print(f"=== Judge Verdict for {item['name']} (via {used_endpoint}) ===")
        print(verdict)
        print()
else:
    print('Skipping OpenCode Go evaluation (add OPENCODE_API_KEY to Colab Secrets to enable).')

In [ ]:
# === Cell 8: Interactive Live-Streaming Prompt Playground (4096+ Token Cap & <think> Enabled) ===
def ask(
    prompt: str,
    max_new_tokens: int = 4096,
    enable_thinking: bool = True,
    reasoning_effort: str = 'high',
    temperature: float = 0.0,
    stream: bool = True,
):
    print(f'[Streaming across {net.total_virtual_layers} virtual layers | max_new_tokens={max_new_tokens} | enable_thinking={enable_thinking}]\n')
    res = generate_sparkx_loop(
        [{'role': 'user', 'content': prompt}],
        max_new_tokens=max_new_tokens,
        enable_thinking=enable_thinking,
        reasoning_effort=reasoning_effort,
        temperature=temperature,
        stream=stream,
    )
    print(f"\n[Speed: {res['tok_per_sec']:.1f} tok/s | Tokens: {res['num_tokens']} | Depth: {net.total_virtual_layers} virtual layers]")
    return res

# Try your own prompt here (streams <think>...</think> + final answer live):
_ = ask('Explain how a 3-tier MoE-Engram memory layer reduces parameter interference during recursive loop finetuning.', max_new_tokens=4096)

In [ ]:
# === Cell 9: Comprehensive 3D Web Page Test — Voxel Cherry Blossom (Sakura) Garden (Three.js / WebGL) ===
import html, re
from pathlib import Path
from IPython.display import display, HTML

SAKURA_3D_PROMPT = (
    "Build a complete, self-contained, single-file 3D interactive web page (`<!DOCTYPE html> ... </html>`) "
    "that renders a **Voxel Cherry Blossom (Sakura) Garden** using Three.js (`https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js`).\n\n"
    "Your single HTML file MUST include all of the following visual & interactive features:\n"
    "1. **Procedural Voxel Island & Pond**: A multi-layered voxel garden island (grass, soil, stone border voxels) with a central tranquil reflective water pond and voxel stepping stones.\n"
    "2. **Traditional Red Voxel Torii Gate or Arched Wooden Bridge**: Spanning across or beside the pond.\n"
    "3. **Procedural Voxel Cherry Blossom (Sakura) Trees**: Multiple gnarled dark-wood voxel trunks and branches crowned with lush multi-tone pink (`#ffb7c5`, `#ff8da1`, `#ffd1dc`, `#fff0f5`) voxel blossom canopies.\n"
    "4. **Japanese Stone Lanterns (`Toro`) with Warm Glowing PointLights**: Placed along the garden path casting warm golden light.\n"
    "5. **Animated Falling Sakura Petal Particle System**: Hundreds of pink voxel petals drifting, swaying in a sinusoidal breeze, rotating in 3D, and respawning when they touch the water/ground.\n"
    "6. **Interactive Camera & Lighting HUD**: Custom mouse-drag orbit + scroll-wheel zoom camera controller, a gentle auto-rotate toggle, and a UI button to cycle between **Spring Daylight**, **Golden Sunset**, and **Lantern Night** atmospheres.\n\n"
    "Output the complete, runnable HTML code inside a single ```html ... ``` code block with zero placeholders or omitted sections."
)

print('=' * 98)
print('RUNNING 3D WEBGL GENERATION TEST: Voxel Cherry Blossom (Sakura) Garden (max_new_tokens=6144)')
print('=' * 98)

sakura_res = generate_sparkx_loop(
    [{'role': 'user', 'content': SAKURA_3D_PROMPT}],
    max_new_tokens=6144,
    enable_thinking=True,
    reasoning_effort='high',
    temperature=0.0,
    repetition_penalty=1.04,
    stream=True,
)
print(f"\n[3D WebGL Generation Stats: {sakura_res['num_tokens']} tokens | {sakura_res['tok_per_sec']:.1f} tok/s | {sakura_res['elapsed_sec']:.1f}s]")

def extract_html_document(text: str) -> str:
    m = re.search(r'```(?:html|HTML)?\s*(<!DOCTYPE html[\s\S]*?</html>)\s*```', text, re.IGNORECASE)
    if m:
        return m.group(1).strip()
    m2 = re.search(r'(<!DOCTYPE html[\s\S]*?</html>|<html[\s\S]*?</html>)', text, re.IGNORECASE)
    if m2:
        return m2.group(1).strip()
    return ''

html_code = extract_html_document(sakura_res['answer']) or extract_html_document(sakura_res['raw_output'])
OUT_HTML_PATH = Path('/content/voxel_cherry_blossom_garden.html')

if html_code:
    OUT_HTML_PATH.write_text(html_code, encoding='utf-8')
    print(f'Saved generated 3D WebGL page to: {OUT_HTML_PATH} ({len(html_code):,} bytes)')

    audit_checks = {
        '1. Valid HTML5 Document (<!DOCTYPE html> ... </html>)': bool(re.search(r'<!DOCTYPE html', html_code, re.I) and re.search(r'</html>', html_code, re.I)),
        '2. Three.js Library Included': ('three' in html_code.lower() and 'THREE' in html_code),
        '3. WebGLRenderer & PerspectiveCamera': ('WebGLRenderer' in html_code and 'PerspectiveCamera' in html_code),
        '4. Voxel BoxGeometry / InstancedMesh': ('BoxGeometry' in html_code or 'InstancedMesh' in html_code),
        '5. Sakura Blossom Palette (Pink/Blossom Hex Colors)': bool(re.search(r'ff[89a-f][0-9a-f]{3}|ffb7c5|ff8da1|ffd1dc|ffc0cb|sakura|blossom', html_code, re.I)),
        '6. Pond / Bridge / Torii / Lantern Voxel Structures': bool(re.search(r'pond|water|bridge|torii|lantern', html_code, re.I)),
        '7. Dynamic Lighting (Ambient/Directional/PointLight)': ('Light' in html_code),
        '8. Falling Petal Animation Loop (requestAnimationFrame)': ('requestAnimationFrame' in html_code),
        '9. Interactive Camera / Mouse / UI Controls': bool(re.search(r'mousedown|pointerdown|wheel|addEventListener|onclick', html_code, re.I)),
        '10. Window Resize Handler': ('resize' in html_code.lower()),
    }
    passed = sum(1 for v in audit_checks.values() if v)
    print(f'\n=== Automated 3D Voxel Sakura Garden Code Audit ({passed}/{len(audit_checks)} Passed) ===')
    for check_name, ok in audit_checks.items():
        print(f'  [{"PASS" if ok else "WARN"}] {check_name}')

    escaped_srcdoc = html.escape(html_code, quote=True)
    preview_iframe = f"""
    <div style="margin-top:12px; border:2px solid #f48fb1; border-radius:12px; overflow:hidden; background:#111;">
      <div style="padding:8px 14px; background:linear-gradient(90deg,#2d1b2e,#4a2545); color:#ffd1dc; font-family:sans-serif; font-size:14px; display:flex; justify-content:space-between; align-items:center;">
        <span>🌸 <b>Spark-X2.6-4B-Loop-mHC (60-Layer)</b> — Live 3D Voxel Cherry Blossom Garden Preview</span>
        <span style="font-size:12px; opacity:0.85;">Drag to Orbit • Scroll to Zoom • Click UI Buttons</span>
      </div>
      <iframe srcdoc="{escaped_srcdoc}" style="width:100%; height:620px; border:none; display:block;" sandbox="allow-scripts allow-same-origin"></iframe>
    </div>
    """
    display(HTML(preview_iframe))
else:
    print('WARNING: Could not extract complete <html>...</html> block; inspect sakura_res["raw_output"] above.')